# CareRisk 48H — Colab training

Quick mode uses only synthetic fixtures and is a smoke test. Full mode uses PhysioNet Set A and produces development candidates. Download/EDA should run on a CPU runtime; enable T4 only after `data/raw` is present on Drive. No cell accesses Set B outcomes.

In [ ]:
MODE = 'quick'  # 'quick' or 'full'
PROJECT_DIR = '/content/drive/MyDrive/CareRisk48H'
DOWNLOAD_SET_A = False  # only set True in a CPU runtime
RESUME = True
assert MODE in {'quick', 'full'}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
project = Path(PROJECT_DIR)
if not (project / 'pyproject.toml').is_file():
    raise FileNotFoundError(f'Place this repository at {project} before running.')
%cd {PROJECT_DIR}

In [ ]:
import subprocess, sys, torch
if DOWNLOAD_SET_A:
    if torch.cuda.is_available():
        raise RuntimeError('Switch Colab to a CPU runtime before downloading/EDA, then return to T4.')
    subprocess.run([sys.executable, 'scripts/download_physionet.py', '--raw-dir', 'data/raw', '--set', 'a'], check=True)
if MODE == 'full' and not (project / 'data/raw/set-a').is_dir():
    raise FileNotFoundError('Full mode needs Set A on Drive. Run the download stage once on CPU.')

In [ ]:
%pip install -q -r requirements-colab.txt
%pip install -q -e . --no-deps
import importlib.metadata as md, json, torch
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:1])) >= (2,)
versions = {name: md.version(name) for name in ['carerisk48h', 'numpy', 'pandas', 'scikit-learn', 'torch', 'lightgbm', 'shap']}
print(json.dumps(versions, indent=2))

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = f'configs/{MODE}.yaml'
checkpoint_dir = project / 'checkpoints' / MODE
checkpoint_dir.mkdir(parents=True, exist_ok=True)
common = ['--config', config, '--device', device, '--checkpoint-dir', str(checkpoint_dir)]
if MODE == 'quick': common += ['--synthetic']
if RESUME: common += ['--resume']
for family in ['grud', 'tcn']:
    subprocess.run([sys.executable, 'scripts/train_deep.py', '--family', family, *common], check=True)

In [ ]:
from datetime import datetime, timezone
artifact_dirs = sorted((project / 'artifacts').glob(f'*-{MODE}'))
for run in artifact_dirs[-2:]:
    required = ['best_model.json', 'preprocessor.npz', 'metrics.json', 'plots']
    print(run.name, {name: (run / name).exists() for name in required})
lock = project / 'artifacts' / f'environment-{MODE}.lock.txt'
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True).stdout
lock.write_text(f'# UTC {datetime.now(timezone.utc).isoformat()}\n' + freeze)
print(f'Environment lock: {lock}')
print('Calibration and model freezing are a separate M5 step; quick runs may never become formal results.')